# 第 3 章 線形回帰

部屋数から住宅価格を予測する線形回帰を、simple / absolute / square の 3 つのトリックで学習します。

対応する記事: [第 3 章 線形回帰（Polyglot Notebook（F#） の言語版）](../../../docs/article/grokking-machine-learning/fsharp/ch03.md)

実装本体: `apps/grokking-ml-fsharp/src/`

## セットアップ

実装本体（`../src/GrokkingMl/`）を `#load` で読み込みます。**ノートブックにコードを複製せず、記事と同じ実装をそのまま使います。**

VS Code の [Polyglot Notebooks 拡張](https://marketplace.visualstudio.com/items?itemName=ms-dotnettools.dotnet-interactive-vscode) で開くか、Jupyter に .NET Interactive カーネルを登録して実行します。

```bash
dotnet tool install -g Microsoft.dotnet-interactive
dotnet interactive jupyter install
jupyter lab notebooks/
```

In [1]:
#load "../src/GrokkingMl/Ch03LinearRegression.fs"

open GrokkingMl.Ch03LinearRegression

## データセット

原著と同じ 6 件の住宅データです。特徴量は部屋数、ラベルは価格です。

In [2]:
let features = [ 1.0; 2.0; 3.0; 5.0; 6.0; 7.0 ]
let labels = [ 155.0; 197.0; 244.0; 356.0; 407.0; 448.0 ]

List.zip features labels
|> List.iter (fun (x, y) -> printfn "部屋数 %.0f → 価格 %.0f" x y)

部屋数 

1

 → 価格 

155

部屋数 

...

407

部屋数 

7

 → 価格 

448

## 3 つのトリックを 1 点分だけ試す

予測が `50 × 3 + 100 = 250`、正解が 300 のとき、それぞれのトリックがどう動くかを見ます。

二乗トリックだけが **誤差の大きさに比例** して動くことに注目してください。

In [3]:
let model = { Slope = 50.0; Intercept = 100.0 }
printfn "予測 %.1f 正解 300.0" (predict model 3.0)

printfn "absolute %A" (absoluteTrick 0.01 model 3.0 300.0)
printfn "square   %A" (squareTrick 0.01 model 3.0 300.0)

予測 

250.0

 正解 300.0

absolute 

{ Slope = 50.03
  Intercept = 100.01 }

square   

{ Slope = 51.5
  Intercept = 100.5 }

## 学習

学習率 0.01、1000 エポックで学習します。真の関係は「傾き 50・切片 100」です。

In [4]:
let trained, errors = linearRegression 0.01 1000 0 features labels

printfn "傾き   %.4f" trained.Slope
printfn "切片   %.4f" trained.Intercept
printfn "RMSE   %.4f" (modelRmse trained features labels)
printfn "初期の RMSE %.2f → 最終 %.4f" (List.head errors) (List.last errors)

傾き   

52.7060

切片   

90.5543

RMSE   

7.0309

初期の RMSE 

316.24

 → 最終 

6.8398

## 誤差の推移

エポックごとの RMSE を 100 エポック刻みで見ます。**最初の数百エポックで大きく下がり、その後は緩やかになります。**

In [5]:
for epoch in 0..100..999 do
    let value = errors[epoch]
    let bar = String.replicate (int (value / 8.0)) "#"
    printfn "epoch %4d  RMSE %7.3f  %s" epoch value bar

epoch 

   0

  RMSE 

316.242

#######################################

...

epoch 

 900

  RMSE 

  8.353

#

## 予測してみる

学習したモデルで、部屋数 4 の家の価格を予測します。

In [6]:
for rooms in [ 1.0; 4.0; 8.0 ] do
    printfn "部屋数 %.0f → 予測価格 %.2f" rooms (predict trained rooms)

部屋数 

1

 → 予測価格 

143.26

部屋数 

...

301.38

部屋数 

8

 → 予測価格 

512.20

## 試してみる

学習率を変えると何が起きるでしょうか。**大きすぎると発散し、小さすぎると収束しません。**

In [7]:
for rate in [ 0.001; 0.01; 0.1 ] do
    let m, _ = linearRegression rate 1000 0 features labels
    printfn "学習率 %-6f 傾き %9.4f  RMSE %10.4f" rate m.Slope (modelRmse m features labels)

学習率 

0.001000

 傾き 

  65.1720

  RMSE 

...

0.100000

 傾き 

529050522173921595138909630555329314434494496768.0000

  RMSE 

1290613420495568898610418412433430910398294392832.0000